In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging
# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
import pickle
from server_config import datapath, preprocessed_path_freezed, redcap_path, preprocessed_path
from functions.preprocessing.ema_mappings import clean_heart_rate_data


In [ ]:
import pandas as pd
import numpy as np
import scipy.stats


In [ ]:
from functions.preprocessing import gps_features
from functions.preprocessing.ema_mappings import run_ema_mappings
from functions.preprocessing.missing_data import summarize_missing_data

In [ ]:
import warnings
# Suppress only SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
df_backup = pd.read_feather(backup_path)
df_backup

In [ ]:
df_backup.type.unique()

In [ ]:
df_backup["delta_t"] = (df_backup["endTimestamp"] - df_backup["startTimestamp"]).dt.total_seconds()
delta_t = df_backup["delta_t"]

In [ ]:
# 1 sec min and 24h max, mostly 60sec
delta_t.describe()

In [ ]:
df_backup.groupby(["customer", "type"])

In [ ]:
# Convert the result to a DataFrame and print as markdown table
result = df_backup.groupby("type")["customer"].nunique().sort_values(ascending=False)
result_df = result.reset_index()
result_df.columns = ['Data Type', 'Unique Customers']
# print(result_df.to_markdown(index=False))
result

In [ ]:
df_backup[df_backup["type"] == "ActiveBurnedCalories"].groupby(
    ["customer", "startTimestamp_day"]
)["doubleValue"].sum().sort_values(ascending=False)

In [ ]:
target_date = pd.to_datetime("2024-05-05")
target_date = pd.to_datetime("2023-12-12")
df_backup[(df_backup["customer"] == "1BAf") & (df_backup["startTimestamp_day"] == target_date)]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["type"] == "ActiveBurnedCalories")]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["startTimestamp_day"] == target_date)]
# df_backup[(df_backup["customer"] == "lAHE") & (df_backup["type"] == "ActiveBurnedCalories")]
# first record says that 17k calories were burned on that day something is off with this 1BAf customer

In [ ]:
df_backup[df_backup["type"] == "HeartRate"]

In [ ]:
(df_backup["startTimestamp"].max() - df_backup["startTimestamp"].min()).total_seconds()

In [ ]:
# df = df_backup[(df_backup["type"] == "HeartRate") & (df_backup["customer"].isin(["4MLe","kVhY"]))]
df = df_backup[(df_backup["type"] == "HeartRate")]
df = df[["customer", "starttimestamp", "endtimestamp", "longValue"]]
df = df.rename(columns={"longValue": "HeartRate"})


In [ ]:
df["duration"] = (df["endtimestamp"] - df["starttimestamp"]).dt.total_seconds().fillna(0).astype(int)

In [ ]:
df_expanded = df.loc[df.index.repeat(df['duration'])].copy()
df_expanded

In [ ]:
df_expanded['time_offset'] = df_expanded.groupby(level=0).cumcount()
df_expanded['timestamp'] = df_expanded['starttimestamp'] + pd.to_timedelta(df_expanded['time_offset'], unit='s')

In [ ]:
df_expanded
df_expanded.groupby(['customer', 'timestamp']).size().value_counts().sort_index()

In [ ]:
# Remove duplicates by taking the average where there are multiple entries per timestamp
df_avg = df_expanded.groupby(['customer', 'timestamp']).agg({
    'HeartRate': 'mean',
}).reset_index()

df_min = df_expanded.groupby(['customer', 'timestamp']).agg({
    'HeartRate': 'min',
}).reset_index()

df_max = df_expanded.groupby(['customer', 'timestamp']).agg({
    'HeartRate': 'max',
}).reset_index()

df_avg

In [ ]:
diff = df_max["HeartRate"] - df_min["HeartRate"]
print(f"proportion of repeated data that have different values: {(diff>0).sum() / (len(df_expanded) - len(df_avg)):.2%}")
print("out of this, the following are the statistics of the differences:")
diff[diff > 0].describe([0.25,0.5,0.75,0.9, 0.95, 0.99])
# TODO resolve what to do with this next, about ~80 of repeated 

In [ ]:
# df = df_avg[df_avg["customer"].isin(["1BAf", "lAHE", "4MLe", "kVhY"])]
df = df_avg

In [ ]:
df["HR_zone_resting"] = df["HeartRate"] < 60
df["HR_zone_moderate"] = (60 <= df["HeartRate"]) & (df["HeartRate"] < 100)
df["HR_zone_vigorous"] = (100 <= df["HeartRate"])

In [ ]:
df

In [ ]:
# Create hourly heart rate averages
df['hour'] = df['timestamp'].dt.floor('h')
df_hourly = df.groupby(['customer', 'hour']).agg(
    HeartRate_count=('HeartRate', 'count'),
    HeartRate_mean=('HeartRate', 'mean'),
    HeartRate_std=('HeartRate', 'std'),
    HeartRate_min=('HeartRate', 'min'),
    HeartRate_max=('HeartRate', 'max'),
    HeartRate_skew=('HeartRate', "skew"),
    HeartRate_kurtosis=('HeartRate', lambda x: x.kurtosis()),
    HeartRate_zone_resting=('HR_zone_resting', 'sum'),
    HeartRate_zone_moderate=('HR_zone_moderate', 'sum'),
    HeartRate_zone_vigorous=('HR_zone_vigorous', 'sum'),
).reset_index()

df_hourly

In [ ]:
# Create daily heart rate averages
df['day'] = df['timestamp'].dt.floor('d')
df_daily = df.groupby(['customer', 'day']).agg(
    HeartRate_count=('HeartRate', 'count'),
    HeartRate_mean=('HeartRate', 'mean'),
    HeartRate_std=('HeartRate', 'std'),
    HeartRate_min=('HeartRate', 'min'),
    HeartRate_max=('HeartRate', 'max'),
    HeartRate_skew=('HeartRate', "skew"),
    HeartRate_kurtosis=('HeartRate', lambda x: x.kurtosis()),
    HeartRate_zone_resting=('HR_zone_resting', 'sum'),
    HeartRate_zone_moderate=('HR_zone_moderate', 'sum'),
    HeartRate_zone_vigorous=('HR_zone_vigorous', 'sum'),
).reset_index()

df_daily